# CS383 — Assignment 1
## Joining and Cleaning Restaurant Inspection Data

*Due: see LMS*

### Scenario
You've been given access to a small relational database of NYC restaurant inspection records, split across three tables:

- `restaurants` — one row per establishment
- `inspections` — one row per inspection visit
- `violations` — one row per violation cited during an inspection

Your task is to write SQL queries that join across these tables, then move into Pandas to clean and summarize what you find.

**This is the first time you're seeing this specific database.** That's intentional — use what you learned about `SELECT`, `WHERE`, `GROUP BY`, joins, and schema-checking (`PRAGMA table_info`, `.dtypes`, `.head()`) to explore it yourself before answering the required questions below. Nothing here has been walked through in class.

---

### Academic integrity and using AI

This assignment is meant to show **your own** understanding of joins and data cleaning — it's the checkpoint that tells you (and your instructor) whether you're ready for Assignment 2 and the capstone, so it needs to reflect what you can actually do without help.

**Allowed** — using an AI tool (Claude, ChatGPT, GitHub Copilot, etc.) as a *tutor*: asking it to explain a concept, explain an error message, explain what a piece of syntax does, or point you toward the right Pandas/SQL method to look up.

**Not allowed** — pasting the assignment questions (or this dataset) into an AI tool and submitting what it gives back; having AI write your queries or code for you; having AI write your reasoning or summary write-ups for you.

A good test: if you can't explain — out loud, without looking anything up — exactly what every line of your submitted notebook does and why, it isn't ready to submit. You may be asked to walk through your code in person; work you can't explain will not receive credit, whether or not AI was involved.

---

### Getting started

Run the cell below once. It builds the three tables above from live NYC Open Data if there's a network connection available, or realistic sample data if not. **Do not skip or modify this cell** — everything after it assumes these three tables exist in `restaurant_inspections.db`.

In [14]:
import os
import sqlite3
import numpy as np
import pandas as pd

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    raw = pd.read_csv(raw_path).head(15000)
    if "score" in raw.columns:
        raw["score"] = pd.to_numeric(raw["score"], errors="coerce")

    restaurants = raw[["camis", "dba", "boro", "cuisine_description"]].drop_duplicates(
        subset="camis"
    ).reset_index(drop=True)

    inspections = raw[["camis", "inspection_date", "action", "score", "grade"]].drop_duplicates(
        subset=["camis", "inspection_date"]
    ).reset_index(drop=True)
    inspections.insert(0, "inspection_id", range(1, len(inspections) + 1))

    with_ids = raw.merge(
        inspections[["camis", "inspection_date", "inspection_id"]], on=["camis", "inspection_date"]
    )
    violations = with_ids.loc[
        with_ids["violation_code"].notna(),
        ["inspection_id", "violation_code", "violation_description", "critical_flag"],
    ].reset_index(drop=True)

    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)

    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza",
                       "Japanese", "Caribbean", "Bakery", "Coffee/Tea", "Chicken"]

    n_restaurants = 150
    camis_ids = np.arange(50_000_001, 50_000_001 + n_restaurants)
    restaurants = pd.DataFrame({
        "camis": camis_ids,
        "dba": [f"Restaurant {i + 1}" for i in range(n_restaurants)],
        "boro": rng.choice(boroughs_list, size=n_restaurants, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n_restaurants),
    })

    # Deliberately messy cuisine text on a chunk of rows (inconsistent case/whitespace).
    messy_idx = rng.choice(restaurants.index, size=20, replace=False)
    def _messify(s):
        pick = rng.integers(0, 3)
        if pick == 0:
            return s.upper()
        elif pick == 1:
            return f"  {s.lower()}  "
        return s.lower()
    restaurants.loc[messy_idx, "cuisine_description"] = restaurants.loc[
        messy_idx, "cuisine_description"
    ].apply(_messify)

    # A handful of exact duplicate restaurant rows.
    restaurants = pd.concat(
        [restaurants, restaurants.sample(5, random_state=1)], ignore_index=True
    )

    n_inspections = 400
    inspections = pd.DataFrame({
        "inspection_id": np.arange(1, n_inspections + 1),
        "camis": rng.choice(camis_ids, size=n_inspections),
        "inspection_date": pd.date_range("2025-01-01", periods=n_inspections, freq="18h").astype(str),
        "action": "Violations were cited in the following area(s).",
        "score": rng.integers(0, 60, size=n_inspections),
        "grade": rng.choice(["A", "B", "C", None], size=n_inspections, p=[0.55, 0.25, 0.10, 0.10]),
    })
    # A few duplicate inspection rows too.
    inspections = pd.concat(
        [inspections, inspections.sample(4, random_state=2)], ignore_index=True
    )

    violation_pool = [
        ("04L", "Evidence of mice or live mice present", "Critical"),
        ("06C", "Food not protected from potential source of contamination", "Critical"),
        ("08A", "Facility not vermin proof", "Not Critical"),
        ("10F", "Non-food contact surface improperly constructed", "Not Critical"),
        ("02B", "Hot food item not held at or above 140 F", "Critical"),
        ("04N", "Filth flies or food/refuse/sewage-associated flies present", "Critical"),
        ("06D", "Food contact surface not properly washed", "Critical"),
        ("05D", "Hand washing facility not accessible", "Critical"),
    ]
    n_violations = 700
    picks = rng.integers(0, len(violation_pool), size=n_violations)
    violations = pd.DataFrame({
        "inspection_id": rng.choice(inspections["inspection_id"].unique(), size=n_violations),
        "violation_code": [violation_pool[i][0] for i in picks],
        "violation_description": [violation_pool[i][1] for i in picks],
        "critical_flag": [violation_pool[i][2] for i in picks],
    })

    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data loaded.")
print(f"restaurants: {len(restaurants):,} rows | inspections: {len(inspections):,} rows | violations: {len(violations):,} rows")

conn = sqlite3.connect("restaurant_inspections.db")
restaurants.to_sql("restaurants", conn, if_exists="replace", index=False)
inspections.to_sql("inspections", conn, if_exists="replace", index=False)
violations.to_sql("violations", conn, if_exists="replace", index=False)

print("Tables ready in restaurant_inspections.db: restaurants, inspections, violations")

Shared snapshot data loaded.
restaurants: 4,279 rows | inspections: 4,529 rows | violations: 14,867 rows
Tables ready in restaurant_inspections.db: restaurants, inspections, violations


---

### Explore before you answer

Before jumping into the required questions, take a few minutes to look around:

- What columns does each table have, and what do they look like? (`PRAGMA table_info(...)`, `.dtypes`, `.head()`)
- How do the three tables relate to each other — which columns would you join on?
- Do you notice anything odd or messy in the data as you look at it?

Use the scratch cell below (and add more cells if you want) to explore. This part isn't graded directly, but it'll make the required questions much easier.

In [15]:
# Scratch space -- explore restaurants, inspections, and violations here.


---

## Required Deliverable 1 — Restaurants Per Cuisine, Cleaned

The `cuisine_description` column has inconsistent capitalization and stray whitespace in places, and the `restaurants` table has a few exact duplicate rows.

Clean both issues in Pandas, then produce a count of restaurants per (cleaned) cuisine type, sorted from most to fewest.

In [16]:
restaurants["cuisine_description"] = (
    restaurants["cuisine_description"].str.strip().str.title())

restaurants = restaurants.drop_duplicates().reset_index(drop=True)

cuisine_counts = (
    restaurants["cuisine_description"].value_counts().rename_axis("cuisine_description").reset_index(name="restaurant_count"))

print(cuisine_counts)

   cuisine_description  restaurant_count
0             American               643
1              Chinese               399
2           Coffee/Tea               379
3                Pizza               238
4       Latin American               196
..                 ...               ...
77             English                 1
78           Ethiopian                 1
79              Afghan                 1
80            Lebanese                 1
81               Cajun                 1

[82 rows x 2 columns]


---

## Required Deliverable 2 — Worst Inspection Scores, With Restaurant Names (JOIN required)

Using a SQL `JOIN` across `restaurants` and `inspections`, find the 10 individual inspections with the **highest** score (a higher score means more violations) among inspections that also received a letter grade.

Your result should include: restaurant name, borough, cuisine, inspection date, score, and grade.

In [17]:
sql = """
SELECT
    r.dba as restaurant_name,
    r.boro as borough,
    r.cuisine_description AS cuisine,
    i.inspection_date,
    i.score,
    i.grade
FROM restaurants as r
JOIN inspections as i
    on r.camis = i.camis
WHERE i.grade IS NOT NULL
ORDER BY i.score DESC
LIMIT 10;
"""

top_10_inspections = pd.read_sql_query(sql, conn)

print(top_10_inspections.to_string(index=False))

                        restaurant_name   borough         cuisine         inspection_date  score grade
              STEINBERG'S KOSHER BAKERY  Brooklyn   Jewish/Kosher 2025-12-23T00:00:00.000  143.0     N
             CARIBBEAN CRAVE RESTAURANT    Queens       Caribbean 2026-01-28T00:00:00.000  131.0     N
                      PANAJACHEL SOLOLA  Brooklyn           Other 2026-08-26T00:00:00.000  130.0     N
PICK AND CHEESE, LAYAWAY BAR AND BODEGA Manhattan           Other 2026-05-26T00:00:00.000  100.0     N
                          INDIA PASSAGE     Bronx     Bangladeshi 2025-11-24T00:00:00.000   96.0     N
                 TOMPKINS SQUARE BAGELS Manhattan           Other 2026-05-26T00:00:00.000   92.0     N
                            CHICK CHICK Manhattan         Chicken 2026-05-26T00:00:00.000   89.0     N
            YUZU MORDERN ASIAN KITCHEN.  Brooklyn   Jewish/Kosher 2025-10-29T00:00:00.000   86.0     C
                    ASIAN HALAL KITCHEN    Queens Southeast Asian 2026-09

---

## Required Deliverable 3 — Most Common Critical Violations, By Cuisine (JOIN required, all 3 tables)

Pick one cuisine type. Using SQL `JOIN`s across all three tables, find the 5 most common `"Critical"` violation descriptions among restaurants of that cuisine.

Then reproduce the same result using Pandas `.merge()` instead of SQL, and confirm the two approaches agree.

In [18]:
sql = """
SELECT
    v.violation_description,
    COUNT(*) AS violation_count
FROM restaurants AS r
JOIN inspections AS i
    ON r.camis = i.camis
JOIN violations AS v
    ON i.inspection_id = v.inspection_id
WHERE r.cuisine_description = 'Chinese'
  AND v.critical_flag = 'Critical'
GROUP BY v.violation_description
ORDER BY violation_count DESC
LIMIT 5;
"""

sql_result = pd.read_sql_query(sql, conn)

print(sql_result)

                               violation_description  violation_count
0  Food, supplies, or equipment not protected fro...              139
1     Hot TCS food item not held at or above 140 °F.              123
2  Cold TCS food item held above 41 °F; smoked or...              121
3  Evidence of mice or live mice in establishment...               81
4  Food contact surface not properly washed, rins...               63


In [19]:
merged = restaurants.merge(inspections, on="camis", how="inner")

merged = merged.merge(violations, on="inspection_id", how="inner")

chinese_critical = merged[
    (merged["cuisine_description"] == "Chinese") &
    (merged["critical_flag"] == "Critical")
]
pandas_answer = (
    chinese_critical["violation_description"].value_counts().head().rename_axis("violation_description").reset_index(name="violation_count")
)

print(pandas_answer)

                               violation_description  violation_count
0  Food, supplies, or equipment not protected fro...              139
1     Hot TCS food item not held at or above 140 °F.              123
2  Cold TCS food item held above 41 °F; smoked or...              121
3  Evidence of mice or live mice in establishment...               81
4  Food contact surface not properly washed, rins...               63


In [20]:
agree = sql_result.equals(pandas_result)

print("SQL and Pandas agree?", agree)

SQL and Pandas agree? True


---

## Required Deliverable 4 — Handle the Missing Grades

Not every inspection has a letter grade — some are still missing (`None` / `NaN`).

Decide how to handle this for the purpose of computing an **average score by grade**: drop the missing rows? Label them `"Ungraded"` and keep them separate? Something else? Write one or two sentences explaining your reasoning *before* your code, then implement it.

I would label missing grades as "Ungraded" rather than dropping them because the inspections still contain valid score information. This keeps all inspections in the analysis while clearly separating inspections that did not receive a letter grade.


In [21]:
inspections["grade"] = inspections["grade"].fillna("Ungraded")

avg_score_by_grade = (inspections.groupby("grade")["score"].mean().reset_index(name="average_score"))

print(avg_score_by_grade)

      grade  average_score
0         A       9.805568
1         B      21.731544
2         C      36.520548
3         N      30.194175
4         P       0.733333
5  Ungraded      31.015067
6         Z      22.547170


---

## Required Deliverable 5 — Summary for a Non-Technical Reader

Pick one finding from your analysis above that an actual restaurant-goer (not a data scientist) would care about.

Produce one clean summary table or chart, and write a two-to-three sentence explanation in plain language — no jargon, no code terms. Imagine you're telling a friend, not writing a lab report.

In [22]:
summary = (inspections.groupby("grade")["score"].mean().reset_index(name="average_score").sort_values("average_score", ascending=False)
)

print(summary)

      grade  average_score
2         C      36.520548
5  Ungraded      31.015067
3         N      30.194175
6         Z      22.547170
1         B      21.731544
0         A       9.805568
4         P       0.733333


**Your summary (plain language):**
    Restaurants with higher average inspection scores had more violations during their inspections. The ungraded inspections are shown separately so they do not get mixed in with restaurants that received an A, B, or C.



---

## Grading Rubric

| Criterion | Points |
|---|---|
| Deliverable 1 — cleaning is correct and counts are right | 15 |
| Deliverable 2 — JOIN is correct and produces the right 10 rows | 20 |
| Deliverable 3 — 3-table JOIN correct; SQL and Pandas results genuinely match | 25 |
| Deliverable 4 — reasoning is stated and implementation matches it | 15 |
| Deliverable 5 — finding is genuinely non-technical, table/chart is clear | 15 |
| Notebook runs top to bottom without errors | 10 |
| **Total** | **100** |

---

## Submission

### One-time GitHub setup (reused for every assignment this semester)

1. On github.com, create one new **public** repository named `cs383-assignments`. You'll keep reusing this same repo all semester — one folder per assignment, not a new repo each time.
2. Inside it, create a folder for this assignment: `assignment1/`.
3. Put your completed notebook in that folder, keeping its original filename: `assignment1_restaurant_inspections.ipynb`.

If you've never pushed to this repo before, from inside your project folder run:

```bash
git init
git remote add origin <your-repo-URL>
git branch -M main
```

(Prefer a GUI? GitHub Desktop or VS Code's built-in Git panel do the same thing without the command line.)

### Every time you submit

```bash
git add -A
git commit -m "Assignment 1: restaurant inspections"
git push
```

Submit the **direct link to your notebook file on GitHub** (not just the repo homepage — the URL should end in `assignment1/assignment1_restaurant_inspections.ipynb`) on BrightSpace under **Assignment 1**.

Make sure your notebook runs top to bottom without errors before you submit — a notebook that only works because cells were run out of order will not be graded as passing.